## Phase 0 — Setup & Data Inspection

Goal: Load all three datasets, understand their structure, check quality, and prepare a clean merged dataset for analysis.

0.1 Environment Setup

Import all libraries (pandas, numpy, matplotlib, seaborn, sklearn, mlxtend, statsmodels, prophet, etc.)
Set display options and plot styles

0.2 Load the Data

Read all three CSVs into DataFrames
Print .shape, .dtypes, .head() for each
Understand what each file represents and how they link via Accident_Index

0.3 Data Quality Audit

Check for missing/null values (.isnull().sum())
Identify sentinel/placeholder values (e.g. -1, 99 often mean "unknown" in UK road data)
Check for duplicate rows
Check data types (dates stored as strings, numeric codes, etc.)

0.4 Data Cleaning

Replace known sentinel values (-1, 99) with NaN
Parse Date column to datetime
Extract Hour from the Time column
Convert Day_of_Week codes to human-readable labels (1=Sunday ... 7=Saturday per UK DfT coding)
Cast categorical columns appropriately

0.5 Merge Datasets

Left-join Accidents → Casualties on Accident_Index
Left-join the result → Vehicles on Accident_Index
Understand the resulting row count (one row per casualty-vehicle combination)
Keep a separate accidents-level DataFrame for tasks that need it

0.6 Exploratory Summary Statistics

.describe() on key numeric columns
Value counts for Accident_Severity, Vehicle_Type, Casualty_Class
Confirm record counts match expected totals

## Phase 1 — Exploratory Data Analysis (Questions 1, 2 & 3)

Goal: Discover time-based and demographic patterns in accidents, motorbike accidents, and pedestrian involvement. Present findings with visualisations and written interpretation.

1.1 Data Cleaning (Phase 1 specific)

- Filter to 2019 records only (confirm all rows are 2019 — cross-check Date)
- Handle nulls in Time, Day_of_Week, Vehicle_Type, Casualty_Class
- Treat outliers in Age_of_Driver, Age_of_Casualty

1.2 Feature Engineering
- Extract Hour_of_Day (0–23) from Time
- Create Day_Name from Day_of_Week code (Mon–Sun)
- Create Time_Band categories: Night (0–6), Morning Rush (7–9), Daytime (10–15), Evening Rush (16–18), Evening (19–23)
- Create Week_Type: Weekday vs Weekend
- Create motorcycle CC categories from Engine_Capacity_(CC):
    - <=125cc
    - 126–500cc
    - > 500cc

1.3 Question 1 — General Accident Patterns (All Accidents)
- Visualisation:
    - Heatmap: Hour of day (y-axis) vs Day of week (x-axis) — count of accidents
    - Bar chart: Total accidents by hour of day
    - Bar chart: Total accidents by day of week
    - Line chart: Accident count by hour, split by weekday vs weekend

Interpretation:
- Identify peak hours and explain the rush-hour effect
- Explain weekend night patterns vs weekday patterns
- Comment on the quietest periods and what they suggest

1.4 Question 2 — Motorbike Accident Patterns
- Data preparation:
    - Filter Vehicle_Type codes for motorcycles (codes 1, 2, 3, 4, 5, 23 in DfT coding)
    - Group into CC categories: <=125cc / 126–500cc / >500cc

- Visualisation:
    - Bar chart: Accidents by CC group and day of week
    - Heatmap: Hour vs Day — one per CC category (3 heatmaps side by side)
    - Line chart: Accidents by hour — all three CC groups overlaid for comparison

- Interpretation: 

    - Compare commuter bikes (<=125cc weekday peak) vs leisure/performance bikes (>500cc weekend peak)
    - Explain why engine size correlates with riding purpose and therefore accident timing

1.5 Question 3 — Pedestrian Accident Patterns
- Data preparation:
    - Filter Casualty_Class == 3 (pedestrian) from the casualties data
    - Merge back to accidents DataFrame to get time columns

- Visualisation:
    - Bar chart: Pedestrian accidents by hour of day
    - Bar chart: Pedestrian accidents by day of week
    - Heatmap: Hour vs Day coloured by pedestrian casualty count
    - Histogram: Age distribution of pedestrian casualties

- Interpretation:
    - Explain school run hours (8–9am, 3–4pm)
    - Discuss evening vulnerability (poor lighting, alcohol)
    - Identify which age groups are most at risk and why

Phase 2 — Association Rule Mining with Apriori (Question 4)

Phase 3 — Spatial Clustering (Question 5)

Phase 4 — Time Series Forecasting by Police Force (Question 6)

Phase 5 — Hull LSOA Daily Accident Forecast (Question 7)

Phase 6 — Final Report & Policy Recommendations


In [ ]:
%pip install ipywidgets

In [ ]:
# Core
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px       # interactive charts
import folium                     # interactive maps

# ML — clustering & association rules
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Time Series
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pmdarima as pm             # auto_arima
from prophet import Prophet

# Evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error